# Step 1 : Install and import Libraries

In [ ]:
# Step 1 : Colab environment setup
# Professional climate-data stack: CDS API, NetCDF/xarray, pandas, NumPy, Cartopy, Matplotlib, R, CDO and NCO
# This version keeps the original capabilities, but avoids the common netCDF4._netCDF4 import failure by using xarray engine fallback.

import os
import sys
import glob
import zipfile
import subprocess
from pathlib import Path
from getpass import getpass

os.environ["DEBIAN_FRONTEND"] = "noninteractive"


def run_setup_command(command, label):
    """Run installation commands quietly, but show useful output if something fails."""
    print(f"Checking/installing: {label} ...")
    result = subprocess.run(
        command,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    if result.returncode != 0:
        print("\n--- Last setup output ---")
        print(result.stdout[-3000:])
        print("\n--- Last setup error ---")
        print(result.stderr[-5000:])
        raise RuntimeError(f"Setup failed while installing/checking: {label}")

    print(f"OK: {label}")


# System tools used later in the notebook
run_setup_command("apt-get -qq update", "apt package index")
run_setup_command("apt-get -qq install -y r-base cdo nco", "R, CDO and NCO")

# Python libraries used throughout the notebook
# h5netcdf/h5py/scipy are intentionally included as robust NetCDF fallback engines.
run_setup_command(
    f"{sys.executable} -m pip install -q --upgrade "
    "cdsapi xarray netCDF4 h5netcdf h5py scipy numpy pandas cartopy matplotlib rpy2 cftime",
    "Python climate and data libraries"
)


import cdsapi
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature


# netCDF4 is useful, but in some Colab sessions the compiled extension can fail.
# If it fails, the notebook will continue using h5netcdf/scipy through xarray.
try:
    import netCDF4
    NETCDF4_AVAILABLE = True
    NETCDF4_VERSION = netCDF4.__version__
    XARRAY_ENGINE_ORDER = ["netcdf4", "h5netcdf", "scipy", None]
except Exception as netcdf4_error:
    netCDF4 = None
    NETCDF4_AVAILABLE = False
    NETCDF4_VERSION = "not available; using xarray fallback engines"
    XARRAY_ENGINE_ORDER = ["h5netcdf", "scipy", None]
    print("\nNote: netCDF4 could not be imported in this Colab runtime.")
    print("The notebook will continue using xarray with h5netcdf/scipy fallback.")
    print("Original netCDF4 import message:", repr(netcdf4_error))


def normalise_time_coordinate(ds):
    """ERA5 files may use valid_time instead of time. Standardise it for the rest of the notebook."""
    if "valid_time" in ds.coords and "time" not in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    return ds


def open_era5_dataset(path):
    """Open a NetCDF file with a safe engine fallback sequence."""
    path = str(path)
    engine_errors = {}

    for engine in XARRAY_ENGINE_ORDER:
        try:
            if engine is None:
                opened = xr.open_dataset(path)
                used_engine = "xarray default"
            else:
                opened = xr.open_dataset(path, engine=engine)
                used_engine = engine

            opened = normalise_time_coordinate(opened)
            opened.attrs["opened_with_engine"] = used_engine
            return opened

        except Exception as error:
            engine_errors[str(engine)] = f"{type(error).__name__}: {error}"

    raise RuntimeError(
        "Could not open the NetCDF file with available xarray engines.\n"
        f"File: {path}\n"
        f"Engine errors: {engine_errors}"
    )


print("\nEnvironment ready.")
print("Python:", sys.version.split()[0])
print("xarray:", xr.__version__)
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("netCDF4:", NETCDF4_VERSION)
print("xarray engine order:", XARRAY_ENGINE_ORDER)

Checking/installing: apt package index ...
OK: apt package index
Checking/installing: R, CDO and NCO ...


# Step 2 : CDS API Setup inside the Google Colab



In [ ]:
# Step 2 : CDS API setup inside Google Colab + Google Drive output folder
# Google Colab is a separate runtime, so it cannot detect the CDS key stored on your local machine.
# This cell also sets the folder where the downloaded ERA5 .nc file will be saved.

from google.colab import drive
from pathlib import Path
from getpass import getpass
import os

# Mount Google Drive
drive.mount("/content/drive")

# Create a project folder inside Google Drive
output_dir = Path("/content/drive/MyDrive/CLIMB_INDIA_ERA5_Demo")
output_dir.mkdir(parents=True, exist_ok=True)

print("Google Drive output folder:")
print(output_dir)

# CDS API token setup
cds_token = getpass("Paste CDS Personal Access Token: ")

cdsapirc_path = Path("/root/.cdsapirc")
cdsapirc_path.write_text(
    "url: https://cds.climate.copernicus.eu/api\n"
    f"key: {cds_token}\n",
    encoding="utf-8"
)

os.chmod(cdsapirc_path, 0o600)

print("CDS API key configured inside this Colab runtime.")

# Step 3 : Download & Process ERA5 Dataset

In [ ]:
# Step 3 : Download & Process ERA5 Data
# Full Pipeline : CDS API → xarray → Health Variable Extraction

import json
import os
import glob
import zipfile
from pathlib import Path

# Ensure Step 2 has created the Google Drive output folder
if "output_dir" not in globals():
    raise NameError(
        "output_dir is not defined. Please run Step 2 first, where Google Drive is mounted "
        "and output_dir is created."
    )

# Download ERA5 variables for India
dataset_name = "reanalysis-era5-single-levels"

target_file = str(output_dir / "era5_india_health.nc")
request_json_file = output_dir / "era5_india_health_request.json"

request = {
    "product_type": "reanalysis",
    "variable": [
        "maximum_2m_temperature_since_previous_post_processing",
        "minimum_2m_temperature_since_previous_post_processing",
        "total_precipitation",
        "2m_dewpoint_temperature"
    ],
    "year": [str(y) for y in range(2020, 2023)],
    "month": [f"{m:02d}" for m in range(1, 13)],
    "day": [f"{d:02d}" for d in range(1, 32)],
    "time": "12:00",
    "area": [37, 68, 6, 98],   # India bbox: North, West, South, East
    "data_format": "netcdf",
    "download_format": "unarchived"
}

# Save request JSON for audit/reproducibility
request_json_file.write_text(
    json.dumps(request, indent=2),
    encoding="utf-8"
)

print("Request JSON saved:", request_json_file)
print("Target NetCDF file:", target_file)

c = cdsapi.Client()

if os.path.exists(target_file):
    print("Existing ERA5 file found in Google Drive. Skipping download:")
    print(target_file)
else:
    try:
        c.retrieve(dataset_name, request, target_file)
        print("Download complete:", target_file)

    except Exception as error:
        print("New CDS request style failed. Trying legacy NetCDF request style...")
        print("Original CDS message:", repr(error))

        legacy_request = request.copy()
        legacy_request.pop("data_format", None)
        legacy_request.pop("download_format", None)
        legacy_request["format"] = "netcdf"

        c.retrieve(dataset_name, legacy_request, target_file)
        print("Download complete using legacy style:", target_file)


# Load ERA5 output
print("\nOpening downloaded ERA5 output...")

print("File exists:", os.path.exists(target_file))
print("File size:", os.path.getsize(target_file), "bytes")

with open(target_file, "rb") as file:
    print("File header:", file.read(8))

era5_nc_files = []

if zipfile.is_zipfile(target_file):
    print("CDS returned a ZIP file. Extracting NetCDF files...")

    extract_dir = output_dir / "era5_india_health_extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(target_file, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

        print("Files inside ZIP:")
        for name in zip_ref.namelist():
            print(" -", name)

    nc_files = sorted(glob.glob(str(extract_dir / "**/*.nc"), recursive=True))
    era5_nc_files = nc_files

    if len(nc_files) == 0:
        raise FileNotFoundError("ZIP extracted, but no .nc files were found.")

    datasets = []

    for file in nc_files:
        temp_ds = open_era5_dataset(file)

        print("\nOpened:", file)
        print("Opened with engine:", temp_ds.attrs.get("opened_with_engine"))
        print("Variables:", list(temp_ds.data_vars))

        datasets.append(temp_ds)

    ds = xr.merge(datasets, compat="override")

else:
    ds = open_era5_dataset(target_file)
    era5_nc_files = [target_file]

print("\nFinal dataset:")
print(ds)
print("\nOpened with engine:", ds.attrs.get("opened_with_engine", "merged dataset"))
print("Available variables:", list(ds.data_vars))


def pick_variable(dataset, logical_name, candidates):
    """Pick a variable robustly using short names and long CDS-style names."""
    for candidate in candidates:
        if candidate in dataset.data_vars:
            print(f"{logical_name}: using variable '{candidate}'")
            return dataset[candidate]

    available = list(dataset.data_vars)
    raise KeyError(
        f"Could not find {logical_name}. Tried: {candidates}. "
        f"Available variables are: {available}"
    )


mx2t = pick_variable(
    ds,
    "maximum 2m temperature",
    ["mx2t", "maximum_2m_temperature_since_previous_post_processing"]
)

mn2t = pick_variable(
    ds,
    "minimum 2m temperature",
    ["mn2t", "minimum_2m_temperature_since_previous_post_processing"]
)

d2m = pick_variable(
    ds,
    "2m dewpoint temperature",
    ["d2m", "2m_dewpoint_temperature"]
)

tp = pick_variable(
    ds,
    "total precipitation",
    ["tp", "total_precipitation"]
)

# Kelvin → Celsius
T = mx2t - 273.15
Tmin = mn2t - 273.15
Td = d2m - 273.15

# Relative humidity approximation from temperature and dewpoint
RH = 100 * np.exp(17.625 * Td / (243.04 + Td)) / np.exp(17.625 * T / (243.04 + T))
RH = RH.clip(0, 100)

# Simplified heat index-style indicator for demonstration
HI = -8.78 + 1.61 * T + 2.34 * RH - 0.146 * T * RH

# Identify spatial dimensions safely
spatial_dims = [dim for dim in ["latitude", "longitude", "lat", "lon"] if dim in T.dims]
if len(spatial_dims) < 2:
    raise ValueError(f"Could not identify latitude/longitude dimensions. T dims are: {T.dims}")

# Count days with maximum temperature > 40°C
heat_days = (T > 40).sum(dim="time")
print(f"Max heat days (grid): {int(heat_days.max().values)}")


# Health Variable Extraction
precipitation_mm = tp * 1000      # metres → millimetres

regional_table = xr.Dataset({
    "max_temperature_c": T.mean(dim=spatial_dims),
    "min_temperature_c": Tmin.mean(dim=spatial_dims),
    "dewpoint_temperature_c": Td.mean(dim=spatial_dims),
    "relative_humidity_percent": RH.mean(dim=spatial_dims),
    "heat_index": HI.mean(dim=spatial_dims),
    "total_precipitation_mm": precipitation_mm.mean(dim=spatial_dims)
}).to_dataframe().reset_index()

regional_table["date"] = pd.to_datetime(regional_table["time"]).dt.date

regional_table = regional_table[
    [
        "date",
        "time",
        "max_temperature_c",
        "min_temperature_c",
        "dewpoint_temperature_c",
        "relative_humidity_percent",
        "heat_index",
        "total_precipitation_mm"
    ]
]

regional_table = regional_table.round(2)

output_csv = output_dir / "era5_india_health_table.csv"
regional_table.to_csv(output_csv, index=False)

display(regional_table.head(10))

print("Saved CSV:", output_csv)
print("Saved NetCDF/ZIP:", target_file)
print("Saved extracted files folder:", output_dir / "era5_india_health_extracted")


# Step 4 : Climate Data Visualisation

In [ ]:
# Python Libraries: Cartopy + Matplotlib

plt.figure(figsize=(11, 4))

plt.plot(
    regional_table["time"],
    regional_table["max_temperature_c"],
    label="Maximum temperature"
)

plt.plot(
    regional_table["time"],
    regional_table["heat_index"],
    label="Heat index"
)

plt.axhline(40, linestyle="--", label="40°C threshold")

plt.title("ERA5 Temperature and Heat Index — India")
plt.xlabel("Time")
plt.ylabel("Value (°C)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.show()


# Map: count days T > 40°C
plt.figure(figsize=(8, 7))

ax = plt.axes(projection=ccrs.PlateCarree())

heat_days.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="Reds",
    cbar_kwargs={"label": "Number of days with T > 40°C"}
)

ax.coastlines()
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)

ax.set_extent([68, 98, 6, 37], crs=ccrs.PlateCarree())
ax.set_title("ERA5 Heat Days Over India: T > 40°C")

plt.show()

# Step 5 : R Workflow inside Colab

In [ ]:
# Step 5A — Activate R inside Google Colab

%load_ext rpy2.ipython

print("R extension loaded. Now R cells can be run using %%R.")

In [ ]:
# Step 5B — Confirm CSV path for R

from pathlib import Path
import os

if "output_csv" in globals():
    csv_path_for_r = str(output_csv)
else:
    csv_path_for_r = str(output_dir / "era5_india_health_table.csv")

print("CSV path for R:")
print(csv_path_for_r)

print("File exists:", os.path.exists(csv_path_for_r))

if not os.path.exists(csv_path_for_r):
    raise FileNotFoundError(
        "CSV file was not found. Run the ERA5 processing cell first."
    )

In [ ]:
# Cell 5C — R should read the CSV file using run_r()

import shutil
import subprocess
import tempfile
import textwrap

# Define run_r here itself, so this cell does not depend on any previous R setup cell
def run_r(r_code, show_output=True, stop_on_error=True):
    """
    Run R code through Rscript.
    This avoids dependency on broken %%R magic or rpy2.
    """
    r_code = textwrap.dedent(r_code)

    if shutil.which("Rscript") is None:
        raise RuntimeError(
            "Rscript is not available. Please run the R setup cell first, or install R in Colab."
        )

    with tempfile.NamedTemporaryFile(mode="w", suffix=".R", delete=False) as f:
        f.write(r_code)
        r_file = f.name

    result = subprocess.run(
        ["Rscript", r_file],
        capture_output=True,
        text=True
    )

    if show_output and result.stdout.strip():
        print(result.stdout)

    if show_output and result.stderr.strip():
        print(result.stderr)

    if stop_on_error and result.returncode != 0:
        raise RuntimeError("R code failed. See the R output above.")

    return result


r_code_5c = r"""
# Cell 5C — R should read the CSV file

csv_path_for_r <- __CSV_PATH__
r_df_rds_path <- __RDS_PATH__

cat("CSV path received by R:\n")
cat(csv_path_for_r, "\n\n")

if (!file.exists(csv_path_for_r)) {
  stop(paste("CSV file not found at:", csv_path_for_r))
}

df <- read.csv(csv_path_for_r)

cat("CSV loaded into R successfully\n")
cat("Rows:", nrow(df), "\n")
cat("Columns:", ncol(df), "\n\n")

print(head(df))

saveRDS(df, file = r_df_rds_path)

cat("\nR dataframe saved for later R cells at:\n")
cat(r_df_rds_path, "\n")
"""

r_code_5c = (
    r_code_5c
    .replace("__CSV_PATH__", csv_path_for_r_safe)
    .replace("__RDS_PATH__", r_df_rds_safe)
)

run_r(r_code_5c)

In [ ]:
# Cell 5D — R summary

import json
import shutil
import subprocess
import tempfile
import textwrap
from pathlib import Path

# Safety: define run_r if it is not already available
if "run_r" not in globals():
    def run_r(r_code, show_output=True, stop_on_error=True):
        """
        Run R code through Rscript.
        This avoids dependency on broken %%R magic or rpy2.
        """
        r_code = textwrap.dedent(r_code)

        if shutil.which("Rscript") is None:
            raise RuntimeError(
                "Rscript is not available. Please run the R setup cell first."
            )

        with tempfile.NamedTemporaryFile(mode="w", suffix=".R", delete=False) as f:
            f.write(r_code)
            r_file = f.name

        result = subprocess.run(
            ["Rscript", r_file],
            capture_output=True,
            text=True
        )

        if show_output and result.stdout.strip():
            print(result.stdout)

        if show_output and result.stderr.strip():
            print(result.stderr)

        if stop_on_error and result.returncode != 0:
            raise RuntimeError("R code failed. See the R output above.")

        return result

# Use the R dataframe saved from Cell 5C
R_DF_RDS = globals().get("R_DF_RDS", "/content/r_state/climate_health_df.rds")

if not Path(R_DF_RDS).exists():
    raise FileNotFoundError(
        "Saved R dataframe not found. Please run Cell 5C first."
    )

r_df_rds_safe = json.dumps(str(R_DF_RDS))

r_code_5d = r"""
# Cell 5D — R summary

r_df_rds_path <- __RDS_PATH__

if (!file.exists(r_df_rds_path)) {
  stop(paste("Saved R dataframe not found at:", r_df_rds_path))
}

df <- readRDS(r_df_rds_path)

cat("Cell 5D — R summary\n")
cat("Rows:", nrow(df), "\n")
cat("Columns:", ncol(df), "\n\n")

summary(df)
"""

r_code_5d = r_code_5d.replace("__RDS_PATH__", r_df_rds_safe)

run_r(r_code_5d)

In [ ]:
# Cell 5E — Prepare a simple demonstration outcome in R using run_r()

import json
import shutil
import subprocess
import tempfile
import textwrap
from pathlib import Path

# Safety: define run_r if it is not already available
if "run_r" not in globals():
    def run_r(r_code, show_output=True, stop_on_error=True):
        """
        Run R code through Rscript.
        This avoids dependency on broken %%R magic or rpy2.
        """
        r_code = textwrap.dedent(r_code)

        if shutil.which("Rscript") is None:
            raise RuntimeError(
                "Rscript is not available. Please run the R setup cell first."
            )

        with tempfile.NamedTemporaryFile(mode="w", suffix=".R", delete=False) as f:
            f.write(r_code)
            r_file = f.name

        result = subprocess.run(
            ["Rscript", r_file],
            capture_output=True,
            text=True
        )

        if show_output and result.stdout.strip():
            print(result.stdout)

        if show_output and result.stderr.strip():
            print(result.stderr)

        if stop_on_error and result.returncode != 0:
            raise RuntimeError("R code failed. See the R output above.")

        return result

# Use the R dataframe saved from Cell 5C and reused in Cell 5D
R_DF_RDS = globals().get("R_DF_RDS", "/content/r_state/climate_health_df.rds")

if not Path(R_DF_RDS).exists():
    raise FileNotFoundError(
        "Saved R dataframe not found. Please run Cell 5C first."
    )

r_df_rds_safe = json.dumps(str(R_DF_RDS))

r_code_5e = r"""
# Cell 5E — Prepare a simple demonstration outcome in R

r_df_rds_path <- __RDS_PATH__

if (!file.exists(r_df_rds_path)) {
  stop(paste("Saved R dataframe not found at:", r_df_rds_path))
}

df <- readRDS(r_df_rds_path)

required_columns <- c("date", "heat_index", "total_precipitation_mm")
missing_columns <- setdiff(required_columns, names(df))

if (length(missing_columns) > 0) {
  stop(paste(
    "Required column(s) missing for Cell 5E:",
    paste(missing_columns, collapse = ", ")
  ))
}

set.seed(42)

df$time_index <- seq_len(nrow(df))

date_parsed <- as.Date(df$date)

if (all(is.na(date_parsed))) {
  warning("The date column could not be parsed fully using as.Date(). dow may contain missing values.")
}

df$dow <- factor(weekdays(date_parsed))

df$death_count <- round(
  20 +
  df$heat_index * 0.15 +
  df$total_precipitation_mm * 0.02 +
  rnorm(nrow(df), mean = 0, sd = 2)
)

cat("Cell 5E — Demonstration outcome prepared in R\n")
cat("Rows:", nrow(df), "\n")
cat("Columns:", ncol(df), "\n\n")

print(head(df))

# Save the updated dataframe for later R cells
saveRDS(df, file = r_df_rds_path)

cat("\nUpdated R dataframe saved for later cells at:\n")
cat(r_df_rds_path, "\n")
"""

r_code_5e = r_code_5e.replace("__RDS_PATH__", r_df_rds_safe)

run_r(r_code_5e)

In [ ]:
# Cell 5F — Basic R visualisation using Rscript, with meaningful colours

import json
import shutil
import subprocess
import tempfile
import textwrap
from pathlib import Path
from IPython.display import Image, display

# ------------------------------------------------------------
# 1. Define safe R runner inside this cell
# ------------------------------------------------------------
def run_r_cell_5f(r_code, show_output=True, stop_on_error=True):
    r_code = textwrap.dedent(r_code)

    if shutil.which("Rscript") is None:
        raise RuntimeError(
            "Rscript is not available. Please run the R setup cell first."
        )

    with tempfile.NamedTemporaryFile(mode="w", suffix=".R", delete=False) as f:
        f.write(r_code)
        r_file = f.name

    result = subprocess.run(
        ["Rscript", r_file],
        capture_output=True,
        text=True
    )

    if show_output and result.stdout.strip():
        print(result.stdout)

    if show_output and result.stderr.strip():
        print(result.stderr)

    if stop_on_error and result.returncode != 0:
        raise RuntimeError("R code failed. See the R output above.")

    return result


# ------------------------------------------------------------
# 2. Use dataframe saved from Cell 5E
# ------------------------------------------------------------
R_DF_RDS = "/content/r_state/climate_health_df.rds"
R_PLOT_PATH = "/content/r_state/cell_5f_heat_index_death_count_coloured.png"

if not Path(R_DF_RDS).exists():
    raise FileNotFoundError(
        "Saved R dataframe not found. Please run Cell 5E first."
    )

Path("/content/r_state").mkdir(parents=True, exist_ok=True)

r_df_rds_safe = json.dumps(R_DF_RDS)
r_plot_path_safe = json.dumps(R_PLOT_PATH)


# ------------------------------------------------------------
# 3. Original R plotting logic, improved with meaningful colours
# ------------------------------------------------------------
r_code_5f = r"""
# Cell 5F — Basic R visualisation with meaningful colours

r_df_rds_path <- __RDS_PATH__
plot_path <- __PLOT_PATH__

df <- readRDS(r_df_rds_path)

required_columns <- c("heat_index", "death_count")
missing_columns <- setdiff(required_columns, names(df))

if (length(missing_columns) > 0) {
  stop(paste(
    "Required column(s) missing for Cell 5F:",
    paste(missing_columns, collapse = ", ")
  ))
}

# Remove rows with missing plotting values
plot_df <- df[!is.na(df$heat_index) & !is.na(df$death_count), ]

if (nrow(plot_df) == 0) {
  stop("No complete rows available for plotting heat_index and death_count.")
}

# Create meaningful heat categories using tertiles
heat_breaks <- quantile(
  plot_df$heat_index,
  probs = c(0, 1/3, 2/3, 1),
  na.rm = TRUE
)

# Make breaks unique in case values are repeated
heat_breaks <- unique(heat_breaks)

if (length(heat_breaks) < 4) {
  # Fallback if quantile breaks are not unique
  plot_df$heat_category <- "Observed heat index"
  point_colours <- "#D95F02"
  legend_labels <- "Observed heat index"
  legend_colours <- "#D95F02"
} else {
  plot_df$heat_category <- cut(
    plot_df$heat_index,
    breaks = heat_breaks,
    include.lowest = TRUE,
    labels = c("Lower heat index", "Moderate heat index", "Higher heat index")
  )

  colour_map <- c(
    "Lower heat index" = "#2C7FB8",
    "Moderate heat index" = "#FDB863",
    "Higher heat index" = "#D7191C"
  )

  point_colours <- colour_map[as.character(plot_df$heat_category)]
  legend_labels <- names(colour_map)
  legend_colours <- colour_map
}

png(
  filename = plot_path,
  width = 950,
  height = 700,
  res = 120
)

# Create base plot
plot(
  plot_df$heat_index,
  plot_df$death_count,
  pch = 19,
  col = point_colours,
  xlab = "Heat Index",
  ylab = "Demonstration Death Count",
  main = "Heat Index and Demonstration Health Outcome"
)

# Add light grid for readability
grid(
  col = "grey85",
  lty = "dotted"
)

# Add a simple linear trend line
trend_model <- lm(death_count ~ heat_index, data = plot_df)
abline(
  trend_model,
  col = "#8B0000",
  lwd = 2
)

# Add legend
legend(
  "topleft",
  legend = c(legend_labels, "Linear trend"),
  col = c(legend_colours, "#8B0000"),
  pch = c(rep(19, length(legend_labels)), NA),
  lty = c(rep(NA, length(legend_labels)), 1),
  lwd = c(rep(NA, length(legend_labels)), 2),
  bty = "n"
)

dev.off()

cat("Cell 5F completed successfully with meaningful colours.\\n")
cat("Blue = lower heat index, orange = moderate heat index, red = higher heat index.\\n")
cat("Plot saved at:\\n")
cat(plot_path, "\\n")
"""

r_code_5f = (
    r_code_5f
    .replace("__RDS_PATH__", r_df_rds_safe)
    .replace("__PLOT_PATH__", r_plot_path_safe)
)

run_r_cell_5f(r_code_5f)


# ------------------------------------------------------------
# 4. Display the R-generated plot in Colab
# ------------------------------------------------------------
display(Image(filename=R_PLOT_PATH))

# Step 6 : CDO/NCO command-line processing

In [ ]:
# Climate Data Operators (CDO) and NetCDF Operators (NCO) for large-scale processing

import os
import glob
import zipfile
from pathlib import Path

if "output_dir" not in globals():
    raise NameError("output_dir is not defined. Run Step 2 first.")

if "target_file" not in globals():
    target_file = str(output_dir / "era5_india_health.nc")

extract_dir = output_dir / "era5_india_health_extracted"

# Build candidate NetCDF file list from Step 3 outputs.
candidate_nc_files = []

if "era5_nc_files" in globals():
    candidate_nc_files.extend([str(path) for path in era5_nc_files])

if extract_dir.exists():
    candidate_nc_files.extend(glob.glob(str(extract_dir / "**/*.nc"), recursive=True))

if os.path.exists(target_file) and not zipfile.is_zipfile(target_file):
    candidate_nc_files.append(str(target_file))

candidate_nc_files = sorted(set(candidate_nc_files))

if len(candidate_nc_files) == 0:
    raise FileNotFoundError("No NetCDF files found. Run Step 3 first.")

print("Candidate NetCDF files:")
for file in candidate_nc_files:
    print(" -", file)


def file_contains_any_variable(path, variables):
    """Check whether a NetCDF file contains any variable in the given list."""
    try:
        temp_ds = open_era5_dataset(path)
        present = [var for var in variables if var in temp_ds.data_vars]
        temp_ds.close()
        return len(present) > 0
    except Exception:
        return False


def find_file_with_variables(variables, label):
    for file in candidate_nc_files:
        if file_contains_any_variable(file, variables):
            return file
    raise FileNotFoundError(
        f"Could not find a NetCDF file containing {label}. "
        f"Tried variables: {variables}"
    )


# These variables may be in one merged file or split across CDS ZIP members.
max_file = find_file_with_variables(["mx2t", "mn2t"], "maximum/minimum temperature variables")
instant_file = find_file_with_variables(["d2m"], "dewpoint variable")
accum_file = find_file_with_variables(["tp"], "precipitation variable")

print("\nMaximum/minimum temperature file:", max_file)
print("Instant/dewpoint file:", instant_file)
print("Accumulated precipitation file:", accum_file)

# Quick variable check before processing
print("\nVariables in temperature file:")
!cdo -s showname "$max_file"

print("\nVariables in dewpoint file:")
!cdo -s showname "$instant_file"

print("\nVariables in precipitation file:")
!cdo -s showname "$accum_file"


# CDO: Compute seasonal mean maximum temperature for India
!cdo -O -seasmean -selvar,mx2t "$max_file" era5_india_seasonal_mx2t.nc

# CDO: Extract specific variable and calculate anomaly
!cdo -O sub -selvar,mx2t "$max_file" -timmean -selvar,mx2t "$max_file" era5_mx2t_anom.nc

# NCO: Convert Kelvin to Celsius
!ncap2 -O -s 'mx2t_c=mx2t-273.15' "$max_file" era5_mx2t_celsius.nc

# CDO: Compute heat wave days (T > 40°C) per year
!cdo -O -yearsum -gtc,40 -selvar,mx2t_c era5_mx2t_celsius.nc heatwave_days.nc

# NCO: Extract temperature variables
!ncks -O -v mx2t,mn2t "$max_file" era5_india_temperature_subset.nc

# NCO: Extract dewpoint variable
!ncks -O -v d2m "$instant_file" era5_india_dewpoint_subset.nc

# NCO: Extract precipitation variable
!ncks -O -v tp "$accum_file" era5_india_precipitation_subset.nc

# CDO: Bilinear interpolation to 0.5° grid
!cdo -O remapbil,r720x360 era5_india_seasonal_mx2t.nc era5_india_0p5deg.nc

print("CDO/NCO processing completed.")
print("Created files:")
!ls -lh *.nc
